# Laboratorium 6

Celem szóstego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmu głębokiego uczenia aktywnego - REINFORCE. Zaimplementowany algorytm będzie testowany z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [39]:
from collections import deque
import gym
import numpy as np
import random
import math

Dołączenie bibliotek do obsługi sieci neuronowych

In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, learning_rate):
        super(DQN, self).__init__()

        self.fc1 = nn.Linear(state_size, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons)
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = np.array(states)
        targets = np.array(targets)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        self.optimizer.step()

## Zadanie 1 - REINFORCE

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu REINFORCE. Wagi sieci aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    \theta \leftarrow \theta + \alpha G_t \nabla_\theta log \pi_{\theta}(a_t, s_t | \theta)
\end{equation*}.
</p>

In [41]:
class REINFORCEAgent:
    def __init__(self, state_size, action_size, model):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.99    # discount rate
        self.learning_rate = 0.001
        self.model = model
        self.state_memory = []
        self.action_memory = []
        self.reward_memory = []
        
        
    def remember(self, state, action, reward):
        #Function adds information to the memory about last action and its results
        self.state_memory.append(state)
        self.action_memory.append(action)
        self.reward_memory.append(reward)

    def get_action(self, state):
        """
        Compute the action to take in the current state, basing on policy returned by the network.

        Note: To pick action according to the probability generated by the network
        """

        q_values = self.model.predict(state.reshape(1, -1)).flatten()
        best_value = np.max(q_values)
        best_actions = np.where(q_values == best_value)[0]
        chosen_action = random.choice(best_actions)
          
        return chosen_action


    def replay(self, batch_size):
        """
        Function learn network using data stored in state, action and reward memory. 
        First calculates G_t for each state and train network
        """
        if len(self.state_memory) == 0 or len(self.action_memory) == 0 or len(self.reward_memory) == 0:
            return
        
        g_t = np.zeros(min(len(self.state_memory), batch_size)).astype(float)

        for index in range(len(self.state_memory)):
            G_t = 0
            power = 0
            for reward in self.reward_memory[index:]:
                G_t += reward * (self.gamma ** power)
                power += 1

            g_t[index] = G_t
            
        self.model.fit(np.array(self.state_memory), np.array(g_t))
        
        
        self.state_memory = []
        self.action_memory = []
        self.reward_memory = []


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [42]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

model = DQN(state_size, action_size, hidden_neurons=24, learning_rate=learning_rate)

c:\Users\Filip\Documents\mgr-siium\G_uczenie_ze_wzmocnieniem\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Przygotuj funkcję obliczającą wartość nagrody skumulowanej:

In [ ]:
def get_cumulative_rewards(rewards,  # rewards at each step
                           gamma=0.99  # discount for reward
                           ):
    """
    based on https://github.com/yandexdataschool/Practica l_RL/blob/spring20/week06_policy_based/reinforce_tensorflow.ipynb
    take a list of immediate rewards r(s,a) for the whole session
    compute cumulative rewards R(s,a) (a.k.a. G(s,a) in Sutton '16)
    R_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...

    The simple way to compute cumulative rewards is to iterate from last to first time tick
    and compute R_t = r_t + gamma*R_{t+1} recurrently

    You must return an array/list of cumulative rewards with as many elements as in the initial rewards.
    """

    cumulative_rewards = np.zeros_like(rewards).astype(float)
    
    for index in reversed(range(len(rewards))):
        cumulative_rewards[index] = rewards[index] 
        if index + 1 < len(rewards):
            cumulative_rewards[index] += gamma * cumulative_rewards[index + 1]        

    return cumulative_rewards


assert len(get_cumulative_rewards(range(100))) == 100
assert np.allclose(get_cumulative_rewards([0, 0, 1, 0, 0, 1, 0], gamma=0.9),
                   [1.40049, 1.5561, 1.729, 0.81, 0.9, 1.0, 0.0])
assert np.allclose(get_cumulative_rewards([0, 0, 1, -2, 3, -4, 0], gamma=0.5),
                   [0.0625, 0.125, 0.25, -1.5, 1.0, -4.0, 0.0])
assert np.allclose(get_cumulative_rewards([0, 0, 1, 2, 3, 4, 0], gamma=0), [0, 0, 1, 2, 3, 4, 0])

Czas nauczyć agenta gry w środowisku *CartPool*:

In [ ]:
agent = REINFORCEAgent(ALPHA=0.001, state_size, action_size, model)


def generate_session(t_max=1000):
    """play env with REINFORCE agent and train at the session end"""

    reward = 0

    s = env.reset()

    for t in range(t_max):

        # chose action
        a = agent.choose_action(s)

        new_s, r, done, info = env.step(a)

        # record session history to train later
        agent.store_transition(s, a, r)

        reward += r

        s = new_s
        if done: break

    agent.learn()

    return reward


for i in range(100):

    rewards = [generate_session() for _ in range(100)]  # generate new sessions

    print("mean reward:%.3f" % (np.mean(rewards)))

    if np.mean(rewards) > 300:
        print("You Win!")
        break

SyntaxError: positional argument follows keyword argument (4058596885.py, line 1)